# Day 4 — ILT 3: Partitioning Strategy, Storage Formats & Performance Levers
### GlobalMart Data Engineering · 2:00 PM – 3:30 PM

---

## Session Objectives

By the end of this session you will be able to:
- Explain why storage format affects query performance
- Compare CSV, JSON, Parquet, and Delta — and choose the right one
- Define partitioning and explain how it enables partition pruning
- Choose the right partition column(s) for a Bronze table
- Explain the dangers of over-partitioning
- Describe Z-ORDER BY and Liquid Clustering, and when to use each
- Identify the small-file problem and fix it with OPTIMIZE / AUTO OPTIMIZE
- Decide when to cache a DataFrame and when not to
- Run VACUUM safely and explain its purpose

---

## Agenda

| Time | Topic |
|------|-------|
| 2:00 | Storage formats — CSV vs JSON vs Parquet vs Delta |
| 2:15 | Partitioning — what it is, partition pruning, choosing columns |
| 2:30 | The over-partitioning trap — the small file problem |
| 2:40 | Z-ORDER BY and Liquid Clustering |
| 2:50 | OPTIMIZE and AUTO OPTIMIZE — compaction |
| 3:00 | Caching — Spark cache vs Databricks disk cache |
| 3:10 | Broadcast joins |
| 3:20 | VACUUM + unified decision framework + Q&A |

> **This session merges two related topics into one continuous story:** how you *lay out* Delta data on disk (format, partitioning, clustering) and how you *keep it fast* once it's there (compaction, caching, joins, cleanup). They're really one decision framework, not two — which is why they're taught together.

---
## 1. Why Storage Format Matters

When Spark reads data, it scans files on ADLS. The format determines:
- **How much data is read** (column pruning, row skipping)
- **How fast it reads** (compression, encoding)
- **What guarantees you get** (ACID, schema enforcement)
- **How much storage you use** (compression ratio)

### The Cost of a Full Scan

```
Bronze orders table: 1.4 million rows, 9 columns

Query: SELECT COUNT(*) FROM bronze_orders WHERE OrderDate = '2026-06-15'

With CSV (no column pruning, no stats):
  → Reads ALL 9 columns for ALL 1.4M rows
  → 100% of data scanned

With Parquet (columnar, compressed):
  → Reads only OrderDate column
  → ~11% of data scanned

With Delta + partitioning (partition pruning):
  → Reads ONLY the partition for 2026-06-15
  → Maybe 0.1% of data scanned
```

> **Format choice is a multiplier on every query cost.**

---
## 2. Storage Format Comparison

| Feature | CSV | JSON | Parquet | Delta |
|---------|-----|------|---------|-------|
| **Type** | Row-based text | Row-based text | Columnar binary | Columnar binary + transaction log |
| **Compression** | None (or gzip) | None | Snappy / ZSTD | Snappy / ZSTD |
| **Schema enforcement** | No | Partial | Yes (in file) | Yes (enforced on write) |
| **Column pruning** | No — reads all columns | No | Yes | Yes |
| **Row skipping (stats)** | No | No | Partial | Yes (min/max per file) |
| **ACID transactions** | No | No | No | Yes |
| **Time travel** | No | No | No | Yes |
| **Schema evolution** | Manual | Manual | Limited | Yes (mergeSchema) |
| **Typical compression ratio** | 1x | 1x | 5–10x | 5–10x |
| **Typical read speed (vs CSV)** | 1x (baseline) | 0.8x | 3–8x | 3–8x |

---

### CSV — The Ingestion Format, Not the Storage Format

```
✅ CSV is good FOR: receiving files from external systems (suppliers, APIs)
❌ CSV is bad FOR: storing data in your Bronze/Silver/Gold layers

Problem with storing in CSV:
  - No schema enforcement → OrderID could be "ORDER-001" or "001" or 1
  - Every query reads every column
  - No statistics → Spark can't skip rows
  - File size bloats without compression
  - No transactions → partial writes leave corrupt data
```

### Parquet — The Columnar Standard

```
✅ Great for read-heavy analytics
✅ Column pruning → only read the columns you need
✅ Compressed → Snappy by default
❌ No ACID → two concurrent writes = corrupt data
❌ No time travel
❌ No easy schema evolution
❌ Updates require rewriting entire files
```

### Delta Lake — Parquet + Transaction Log

```
Delta = Parquet files + _delta_log/ folder

_delta_log/:
  00000000000000000000.json  ← Version 0: initial table creation
  00000000000000000001.json  ← Version 1: first write
  00000000000000000002.json  ← Version 2: OPTIMIZE run
  ...
  00000000000000000010.checkpoint.parquet  ← periodic checkpoint

Each log entry records:
  - Which files were added
  - Which files were removed
  - Min/max statistics per column per file
  - Schema at this version
  - Who made the change and when
```

**Delta = Parquet performance + ACID + time travel + schema evolution**

> **Use Delta for ALL Bronze, Silver, and Gold tables. No exceptions.**

---
## 3. Why Delta Is the Right Choice for Bronze

### 1. ACID Guarantees

```
Scenario: Auto Loader is mid-write to Bronze when the cluster dies.

Without ACID (Parquet/CSV):
  → Partial file written → 50,000 rows half-written → data corrupt
  → No way to know which rows made it

With Delta:
  → Transaction was not committed → Delta treats it as if it never happened
  → Next run restarts from checkpoint → clean, correct state
```

### 2. Time Travel

```sql
-- Read Bronze as it was yesterday (before a bad batch)
SELECT COUNT(*) FROM gbmart.bronze.orders
VERSION AS OF 5;

-- Or by timestamp:
SELECT * FROM gbmart.bronze.orders
TIMESTAMP AS OF '2026-06-10 12:00:00';
```

### 3. Schema Evolution

```python
# New column in source? Just add mergeSchema:
df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("gbmart.bronze.products")
# Bronze table automatically gains the new column
```

### 4. File Statistics for Query Optimisation

```
Delta stores min/max per column per file in the transaction log.

Query: WHERE order_date = '2026-06-15'

Delta checks transaction log:
  File A: order_date min=2026-06-01, max=2026-06-14 → SKIP
  File B: order_date min=2026-06-15, max=2026-06-20 → READ
  File C: order_date min=2026-06-21, max=2026-06-30 → SKIP

Result: only File B is read → massive I/O saving
```

This is called **data skipping** — and it's automatic with Delta.

---
## 4. What Is Partitioning?

Partitioning = physically splitting a table into **sub-folders** based on a column's value.

```
Without partitioning:
gbmart.bronze.orders (underlying files):
  part-00000.parquet   ← all 1.4M rows in one folder
  part-00001.parquet
  part-00002.parquet

With partitioning by ingestion_date:
gbmart.bronze.orders (underlying files):
  _ingested_at=2026-06-13/
    part-00000.parquet   ← only rows from June 13
  _ingested_at=2026-06-14/
    part-00000.parquet   ← only rows from June 14
  _ingested_at=2026-06-15/
    part-00000.parquet   ← only rows from June 15
```

### Partition Pruning

```
Query: SELECT * FROM gbmart.bronze.orders WHERE ingestion_date = '2026-06-15'

With partitioning:
  Spark looks at folder structure
  → Finds ingestion_date=2026-06-15/
  → Reads ONLY that folder
  → Skips ingestion_date=2026-06-13/ and ingestion_date=2026-06-14/

Result: if you have 30 days of data, Spark reads 1/30 = 3.3% of data
```

### How to Write a Partitioned Delta Table (Unity Catalog managed table)

```python
from pyspark.sql.functions import to_date, current_timestamp

df = df.withColumn("ingestion_date", to_date(current_timestamp()))

df.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("ingestion_date") \
    .saveAsTable("gbmart.bronze.orders")
```

---
## 5. Choosing Partition Columns

### The 3 Criteria for a Good Partition Column

1. **High query filter frequency** — it's in your WHERE clause almost every time
2. **Low-to-medium cardinality** — not too many unique values
3. **Evenly distributed** — data spread roughly equally across partitions

---

### Good Partition Column Choices for Bronze

| Column | Cardinality | Good? | Reason |
|--------|------------|-------|--------|
| `ingestion_date` | ~365 values/year | ✅ Yes | Most queries filter by date; daily batch → natural partition |
| `order_channel` | 3-5 values | ✅ Yes | Low cardinality; if frequently filtered |
| `category` (products) | 10-50 values | ✅ Maybe | Only if you often query per category |
| `customer_id` | Tens of thousands | ❌ No | Too high — thousands of tiny partitions |
| `order_id` | Millions | ❌ No | Never partition on a primary key |
| `order_date` | Thousands | ❌ Careful | Use only if daily queries and data volume justifies it |

### GlobalMart Bronze Partitioning Strategy

```
gbmart.bronze.orders:      would use partitionBy("ingestion_date") IF volume justified it
gbmart.bronze.order_items: same
gbmart.bronze.products:    no partitioning — reference table, small, Z-ORDER instead
gbmart.bronze.customers:   no partitioning — reference table, small, Z-ORDER instead
```

> **For Bronze, `ingestion_date` is almost always the right partition column IF the table is large enough to justify it.** Queries against Bronze typically ask: "show me what was ingested on date X" — but see the over-partitioning trap below before applying this to every table.

---
## 6. The Over-Partitioning Trap

Partitioning creates one folder per unique value. Too many folders = **the small file problem**.

### What Happens When You Over-Partition

```
Scenario: Partition bronze_adls_orders by OrderID
  1,400,000 rows → 1,400,000 partitions → 1,400,000 folders
  Each folder: 1 file with 1 row

Problems:
  1. Metadata overhead: Delta transaction log tracks 1.4M files → slow to open table
  2. Spark task overhead: 1 Spark task per partition = 1.4M tasks for a simple COUNT
  3. ADLS list calls: listing 1.4M folders takes minutes
  4. Small files: ADLS is optimised for large files (128MB+), not tiny 1-row files
```

### The Small File Problem

```
Ideal file size: 128 MB – 1 GB (for Parquet/Delta on ADLS)

Small file scenario:
  1000 files × 100 KB = 100 MB total data
  vs
  1 file × 100 MB = 100 MB total data

Reading 1000 small files:
  → 1000 ADLS open() calls
  → 1000 Spark tasks
  → Overhead dominates actual data processing time
  → Queries are slow despite small data volume

Reading 1 large file:
  → 1 ADLS open() call
  → 1 Spark task (or few with parallelism)
  → Fast
```

### The Rule of Thumb

```
Minimum partition size: 1 GB of data per partition value

If your daily ingestion is 50 MB → do NOT partition by date
If your daily ingestion is 5 GB  → partitioning by date makes sense
```

> For GlobalMart's 1.4M orders, if the full dataset is ~200 MB, partitioning by date
> across 365 days = ~550 KB per partition. Too small. Better: no partitioning + Z-ORDER.

---
## 7. Z-ORDER BY — Clustering Without Partitioning

Z-ORDER is Delta's way of co-locating related rows within files, without creating sub-folders.

### How Z-ORDER Works

```
BEFORE OPTIMIZE ... ZORDER BY (customer_id):
  File A: customer_id = C001, C045, C200, C999 (random mix)
  File B: customer_id = C002, C100, C500, C700 (random mix)

  Query: WHERE customer_id = 'C001'
  → Must scan both File A and File B to be sure

AFTER OPTIMIZE ... ZORDER BY (customer_id):
  File A: customer_id = C001 – C250  (low range)
  File B: customer_id = C251 – C500  (mid range)
  File C: customer_id = C501 – C999  (high range)

  Query: WHERE customer_id = 'C001'
  → Delta stats say: File A min=C001, File B min=C251 → skip B and C
  → Only File A read → much faster
```

### Running OPTIMIZE + ZORDER

```sql
-- Run after bulk loads or periodically — Unity Catalog table name, no path needed
OPTIMIZE gbmart.bronze.orders
ZORDER BY (customer_id, order_date);
```

```python
# Or using Python:
from delta.tables import DeltaTable

dt = DeltaTable.forName(spark, "gbmart.bronze.orders")
dt.optimize().executeZOrderBy(["customer_id", "order_date"])
```

### Z-ORDER vs Partitioning

| | Partitioning | Z-ORDER |
|--|-------------|----------|
| Creates sub-folders | Yes | No |
| Query pruning | Folder-level (partition pruning) | File-level (data skipping via stats) |
| Risk of small files | High (if over-partitioned) | Low |
| Cardinality limit | Low-medium cardinality only | Works with high cardinality |
| Cost | Free (at write time) | Requires OPTIMIZE run |
| Best for | Date/region/status | customer_id, product_id, order_id |

---
## 8. Liquid Clustering (Delta 3.0+)

Liquid Clustering is the next evolution — it replaces both partitioning AND Z-ORDER with a unified, adaptive approach.

### The Problem With Static Partitioning

```
You partitioned by order_channel (3 values: Online, Mobile, In-Store)

6 months later:
  - Business starts filtering by category, not order_channel
  - To change partition columns → must rewrite the entire table
  - Changing partitions on a 10TB table = hours of compute + storage
```

### Liquid Clustering Solution

```sql
-- Enable when creating the table:
CREATE TABLE gbmart.bronze.orders
CLUSTER BY (customer_id, order_date);

-- Change clustering columns any time — NO full rewrite:
ALTER TABLE gbmart.bronze.orders
CLUSTER BY (order_channel, order_date);  -- changed, no data rewrite

-- Incrementally cluster (only uncompacted files):
OPTIMIZE gbmart.bronze.orders;  -- same command, no ZORDER needed
```

### Liquid Clustering vs Z-ORDER vs Partitioning

| | Partitioning | Z-ORDER | Liquid Clustering |
|--|-------------|---------|-------------------|
| Column change | Full rewrite | Easy | Easy |
| Small file risk | High | Low | Low |
| Incremental | No | No | Yes |
| Maintenance | Manual OPTIMIZE | Manual OPTIMIZE | OPTIMIZE (same cmd) |
| Availability | All Delta versions | All Delta versions | Delta 3.0+ / DBR 13.3+ |
| Best for | Simple date partitions on large tables | Medium cardinality columns | Most new tables in Databricks |

> **Recommendation for new tables on Databricks:** Use Liquid Clustering. For older tables or simple date-based splits on very large tables, partitioning still works well.

---

## 9. Layout Decision Framework

```
What is my daily data volume per partition?
      |
      ├── < 1 GB per day
      │       → DO NOT partition
      │       → Use Z-ORDER or Liquid Clustering on query columns
      │
      └── > 1 GB per day
              |
              ├── Query pattern is date-based, low cardinality
              │       → Partition by ingestion_date or date column
              │       → Add Z-ORDER for secondary filter columns
              │
              └── Query pattern is multi-dimensional, may change
                      → Use Liquid Clustering (if DBR 13.3+)
```

The next sections shift from *layout* (how data is organized on disk) to *performance levers* (how you keep queries fast day to day) — small files, caching, joins, and safe cleanup.

---
## 10. The Small File Problem

### How Small Files Are Created

```
Streaming writes (every 30 seconds):
  Trigger 1: 500 rows → 1 small Parquet file (~50 KB)
  Trigger 2: 500 rows → 1 small Parquet file (~50 KB)
  ...
  Trigger 1440: 500 rows → 1 small Parquet file (~50 KB)

After 24 hours of streaming:
  1,440 files × 50 KB = 72 MB total data
  But: 1,440 storage-layer open() calls for any full scan
       1,440 Spark tasks minimum
       1,440 entries in Delta transaction log
```

### Symptoms of Small Files

| Symptom | What You See |
|---------|-------------|
| Slow queries despite small data | A 100 MB table takes 30 seconds to scan |
| High task count | Spark shows 1,000+ tasks for a simple COUNT |
| Slow table open | `spark.table('gbmart.bronze.orders')` takes a long time just to start |
| Delta log bloat | `_delta_log/` has thousands of JSON files |

### Why Cloud Storage Hates Small Files

```
ADLS (Azure Data Lake Storage) is optimised for:
  ✅ Sequential reads of large files (128 MB – several GB)
  ✅ High throughput per file

ADLS is slow for:
  ❌ Many small files — each open() has network round-trip latency
  ❌ Listing thousands of files — metadata operations are expensive

Ideal file size: 128 MB – 1 GB per file
Below 32 MB:     starting to become a problem
Below 1 MB:      definitely a problem
```

---
## 11. OPTIMIZE — Manual Compaction

OPTIMIZE reads many small files and rewrites them into fewer, larger files.

```
BEFORE OPTIMIZE:
  1,440 files × 50 KB = 72 MB

AFTER OPTIMIZE:
  1 file × 72 MB
  (or a few files if parallelism writes multiple)
```

### Running OPTIMIZE

```sql
-- Basic compaction — Unity Catalog table name, no path needed:
OPTIMIZE gbmart.bronze.orders;

-- Compaction + Z-ORDER (co-locate by column for faster filtering):
OPTIMIZE gbmart.bronze.orders
ZORDER BY (customer_id, order_date);

-- Compact only a specific partition (faster for large tables):
OPTIMIZE gbmart.bronze.orders
WHERE ingestion_date = '2026-06-15';
```

```python
# Python equivalent:
from delta.tables import DeltaTable

dt = DeltaTable.forName(spark, "gbmart.bronze.orders")
dt.optimize().executeCompaction()

# With Z-ORDER:
dt.optimize().executeZOrderBy(["customer_id", "order_date"])
```

### What OPTIMIZE Does NOT Do

```
OPTIMIZE:
  ✅ Rewrites small files into large files
  ✅ Updates Delta transaction log (new files added, old files marked removed)
  ✅ Optionally Z-ORDERs data within files

OPTIMIZE does NOT:
  ❌ Delete the old small files (they stay, just marked as removed in log)
  ❌ Free up storage (need VACUUM for that — see later)
  ❌ Change the data — same rows, just reorganised into larger files
```

### When to Run OPTIMIZE

```
✅ After every large batch load
✅ Daily on streaming Bronze tables (schedule in Databricks Workflows)
✅ After a bulk DELETE or UPDATE on Silver/Gold
✅ Before a big analytical query that will run repeatedly

❌ Not necessary if using AUTO OPTIMIZE (below)
❌ Not on tables that are constantly being appended to mid-query
```

---
## 12. AUTO OPTIMIZE — Automatic Compaction

Instead of manually running OPTIMIZE, you can tell Delta to compact files automatically during writes.

### Two Settings

```python
# Option 1: Set on the Spark session (affects all Delta writes in this session)
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled",  "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled",    "true")

# Option 2: Set as Delta table properties (persists — affects all future writes)
spark.sql("""
    ALTER TABLE gbmart.bronze.orders
    SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true'
    )
""")
```

### What Each Setting Does

| Setting | What It Does |
|---------|-------------|
| `optimizeWrite` | Before writing, Spark shuffles data to reduce the number of output files. Instead of 200 tasks × 1 file each = 200 small files, it targets ~128 MB per output file. |
| `autoCompact` | After a write completes, Delta runs a background OPTIMIZE on the written files. Happens asynchronously — doesn't slow down the write. |

### AUTO OPTIMIZE vs Manual OPTIMIZE

```
autoOptimize.optimizeWrite:
  → Prevents small files from being created in the first place
  → Happens AT write time
  → Slightly increases write time (shuffle overhead)

autoOptimize.autoCompact:
  → Compacts files AFTER they are written
  → Background operation
  → Does not include Z-ORDER (for Z-ORDER, still need manual OPTIMIZE)
```

> **Recommendation for Bronze streaming tables:** Enable both AUTO OPTIMIZE settings.
> For Z-ORDER, still schedule a daily manual `OPTIMIZE ... ZORDER BY` job (Databricks Workflows — covered later in the course).

---
## 13. Caching

Caching stores a DataFrame's data in memory (or on disk) so subsequent operations re-read from cache instead of re-scanning storage.

### Spark Cache (In-Memory)

```python
# Cache a DataFrame in memory:
bronze_df = spark.table("gbmart.bronze.orders")
bronze_df.cache()

# First access — reads from storage, stores in memory:
bronze_df.count()  # triggers caching

# Second access — reads from memory:
bronze_df.count()   # much faster
bronze_df.filter(...).show()  # also from cache

# Free memory when done:
bronze_df.unpersist()
```

### Storage Levels

```python
from pyspark import StorageLevel

# Memory only (default .cache()):
df.persist(StorageLevel.MEMORY_ONLY)

# Memory + disk (spills to disk if memory full):
df.persist(StorageLevel.MEMORY_AND_DISK)

# Disk only (useful for DataFrames too large for memory):
df.persist(StorageLevel.DISK_ONLY)
```

### Databricks Disk Cache (Automatic)

```
Databricks has a second layer of caching called the "Disk Cache" (formerly Delta Cache):

- Operates at the STORAGE LAYER — caches raw Parquet data from cloud storage
- Enabled automatically on some cluster types (General Purpose, Storage Optimized)
- Transparent — no code changes needed
- Persists ACROSS Spark jobs on the same cluster
- Survives notebook detach/reattach (unlike Spark cache)

Spark cache vs Databricks disk cache:
  Spark cache:     you control it (.cache(), .unpersist())
                   stored as Spark RDD partitions in JVM heap
                   lost when cluster restarts

  Databricks cache: automatic
                    stored on local SSDs of cluster nodes
                    survives across jobs, faster for large scans
```

### When to Cache, When Not To

| Scenario | Cache? | Reason |
|----------|--------|--------|
| Small reference table used in multiple joins | ✅ Yes | Avoids re-reading from storage on each join |
| Large Bronze table for a single scan | ❌ No | Cache won't fit in memory; single scan doesn't benefit |
| A dimension table iterated across many Gold queries | ✅ Yes | Massive saving — many reads → 1 read |
| Streaming source | ❌ No | Streaming state managed differently |
| Iterative aggregations on the same DF | ✅ Yes | Each `groupBy` would re-read from disk without cache |

---
## 14. Broadcast Joins

When joining a large table to a small table, tell Spark to broadcast the small table to all workers instead of doing a sort-merge join.

### Sort-Merge Join (Default — Expensive)

```
Large table: gbmart.bronze.orders (1.4M rows)
Small table: gbmart.gold.dim_payment_method (5 rows)

Default sort-merge join:
  1. Shuffle ALL of orders across network (1.4M rows moved)
  2. Shuffle ALL of dim_payment_method across network (5 rows moved)
  3. Sort both sides
  4. Merge

Problem: shuffling 1.4M rows across the network is expensive
```

### Broadcast Join (Small Table → All Workers)

```python
from pyspark.sql.functions import broadcast

# Option 1: Explicit broadcast hint:
result = orders_df.join(
    broadcast(dim_payment_method_df),
    on="payment_method_id",
    how="left"
)

# Option 2: Let Spark decide automatically (if table < autoBroadcastJoinThreshold):
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "50mb")  # default 10MB
```

```
Broadcast join:
  1. Ship entire dim_payment_method (5 rows) to ALL worker nodes once
  2. Each worker joins its local partition of orders to its local copy
  3. No network shuffle of orders at all

Result: 100x+ faster for small-to-large joins
```

### When to Use Broadcast

```
Use broadcast when:
  ✅ One table is small (< 100 MB — adjust threshold if needed)
  ✅ The small table is used repeatedly (broadcast once, reuse)
  ✅ Joining lookup/reference tables: dim_payment_method, dim_date, dim_product

Do NOT broadcast when:
  ❌ Both tables are large (both need to shuffle → no benefit)
  ❌ Table is > 1 GB (driver runs out of memory collecting it)
```

---
## 15. VACUUM — Cleaning Up Old Files

Recall: OPTIMIZE marks old small files as "removed" in the Delta log but does NOT delete them from storage.

VACUUM physically deletes those removed files to free storage.

### Running VACUUM

```sql
-- Delete files older than 7 days (default) — Unity Catalog table name, no path needed:
VACUUM gbmart.bronze.orders;

-- Custom retention (minimum 7 days recommended):
VACUUM gbmart.bronze.orders RETAIN 168 HOURS;  -- 7 days

-- Dry run (shows what WOULD be deleted without actually deleting):
VACUUM gbmart.bronze.orders RETAIN 168 HOURS DRY RUN;
```

### The 7-Day Retention Rule

```
Default retention: 7 days (168 hours)

Why 7 days minimum?
  Delta time travel works by reading old file versions from the transaction log.
  If you VACUUM files that are still referenced by recent versions,
  time travel to those versions will fail with FileNotFoundException.

  7 days = safe window for most use cases

Dangerous:
  VACUUM gbmart.bronze.orders RETAIN 0 HOURS;
  → Deletes ALL previous version files immediately
  → Time travel is permanently disabled for this table
  → Cannot be undone
```

### VACUUM vs OPTIMIZE — Summary

```
OPTIMIZE:   Compacts files (rewrites small → large)
             Old files marked removed in log
             Storage NOT freed

VACUUM:     Deletes files marked as removed by OPTIMIZE (or DELETE/UPDATE)
            Storage IS freed
            Time travel to deleted versions is no longer possible

Typical maintenance schedule:
  Daily:   OPTIMIZE (compaction + Z-ORDER)
  Weekly:  VACUUM RETAIN 168 HOURS (free storage, keep 7-day time travel)
```

> **Safety note for this course:** running `VACUUM ... DRY RUN` is harmless and free to try. Running a real `VACUUM` or `OPTIMIZE` against `gbmart.bronze.*` tables touches shared, real training data used by the whole cohort — check with your instructor before running either against shared tables outside your own personal schema/catalog.

---
## 16. Unified Decision Framework

```
PERFORMANCE PROBLEM?
      |
      ├── Query is slow on a large table
      │       |
      │       ├── Many small files?  → OPTIMIZE (compact) + AUTO OPTIMIZE (prevent)
      │       ├── Filtering by a column?  → ZORDER BY that column, or Liquid Clustering
      │       ├── Joining small table?  → broadcast() the small table
      │       └── Scanning same data repeatedly?  → .cache() the DataFrame
      │
      ├── Storage cost is too high
      │       → VACUUM RETAIN 168 HOURS (delete old file versions)
      │
      ├── Write is slow (many small output files)
      │       → Enable optimizeWrite = true
      │
      └── Choosing a layout for a brand-new table
              → < 1 GB/day: Z-ORDER or Liquid Clustering, no partitioning
              → > 1 GB/day, date-filtered: partition by date + Z-ORDER secondary columns
              → Multi-dimensional / evolving query patterns: Liquid Clustering
```

---

## Key Takeaways

1. **Always use Delta** for Bronze/Silver/Gold — not CSV, not Parquet alone
2. **Partitioning = folder-level pruning** — great for date columns on large tables; over-partitioning causes the small-file problem
3. **Z-ORDER / Liquid Clustering = file-level skipping** — for high-cardinality columns without folder explosion; Liquid Clustering is the modern default
4. **Small files are the #1 performance issue** in streaming Bronze pipelines — `OPTIMIZE` fixes it, `AUTO OPTIMIZE` prevents it
5. **Cache** small reference tables and DataFrames used repeatedly — not large tables scanned once
6. **Broadcast** the small side of every join under ~100 MB
7. **VACUUM** weekly to free storage — always keep the 7-day retention minimum, or you lose time travel permanently

---

## Discussion Questions

1. *Bronze table has 200 MB of data. A student suggests partitioning by `ingestion_date`. What's the problem, and what should they use instead?*

2. *A Bronze table has 10,000 files averaging 30 KB each. What command fixes this? What does it actually do to the data?*

3. *After running OPTIMIZE, storage usage hasn't gone down. Why? What command do you need, and what's the risk of running it carelessly?*

4. *A Silver job joins `gbmart.bronze.orders` (500M rows) with `gbmart.gold.dim_product` (2,000 rows). What join optimisation should you apply and why?*

5. *What is the main advantage of Liquid Clustering over static partitioning when query patterns change?*